# OpenPI pi0.5 xArm LoRA Fine-Tuning Pipeline

This notebook is organized around one rule: Google Drive is the persistent state, and Colab `/content` is disposable runtime state. Re-running after a runtime reconnect should reuse raw data, converted LeRobot data, norm stats, base weights, checkpoints, and final exports from Drive whenever the inputs have not changed.

Persistent Drive layout:

```text
MyDrive/embodied_ai_xarm/
  raw/                  # raw task/episode folders, or extracted raw.zip
  raw.zip               # optional source archive containing raw/
  raw/*.zip             # optional per-task archives, e.g. raw/pick_up_the_red_pepper.zip
  code/                 # fine_tune helper scripts copied into /content each runtime
  lerobot/              # persistent LeRobotDataset cache
  openpi_cache/         # persistent pi05_base checkpoint
  openpi_assets/        # persistent OpenPI norm stats
  openpi_checkpoints/   # persistent training checkpoints
  final_models/         # inference exports
  pipeline_state/       # small manifests used to skip unchanged work
```

## How to Re-Run

For a fresh Colab runtime, run cells 1 through 6, then run training. If raw data and converter/config did not change, conversion and norm-stat cells will restore from Drive instead of recomputing.

Use the force flags in cell 1 only when you intentionally want to redo a step. If you replace raw images in Drive without changing `raw.zip` or robot logs, bump `RAW_DATA_VERSION` in cell 1 so cached conversion/norm stats are invalidated.

W&B defaults to offline mode for stability. Set `WANDB_MODE_SETTING = 'online'` in cell 1 only after authenticating with `wandb.login()`.

## 1. Mount Drive and Configure Persistent Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import zipfile

DRIVE_ROOT = Path('/content/drive/MyDrive/embodied_ai_xarm')
RAW_ZIP = DRIVE_ROOT / 'raw.zip'
RAW_ROOT = DRIVE_ROOT / 'raw'
CODE_ROOT = DRIVE_ROOT / 'code'
PROJECT_DIR = Path('/content/embodied-ai-xarm')
OPENPI_DIR = Path('/content/openpi')
HF_LEROBOT_HOME = DRIVE_ROOT / 'lerobot'
PI05_BASE_PARAMS = DRIVE_ROOT / 'openpi_cache/pi05_base/params'
ASSETS_BACKUP = DRIVE_ROOT / 'openpi_assets'
CHECKPOINT_BACKUP = DRIVE_ROOT / 'openpi_checkpoints'
FINAL_MODELS = DRIVE_ROOT / 'final_models'
STATE_DIR = DRIVE_ROOT / 'pipeline_state'

CONFIG_NAME = 'pi05_xarm_full_finetune'
EXP_NAME = 'pi05_xarm_full_finetune'
SMOKE_CONFIG_NAME = 'pi05_xarm_colab_smoke'
REPO_ID = 'local/xarm_pi05_data'
# Override this if your uploaded LeRobotDataset lives somewhere else on Drive.
LEROBOT_DATASET_ROOT = HF_LEROBOT_HOME / REPO_ID
ROBOT_TYPE = 'xarm6'
FPS = 10
# Bump this when replacing raw images in Drive without changing raw.zip or robot logs.
RAW_DATA_VERSION = '2026-07-raw-v1'

# Default behavior: reuse Drive state. Flip one of these only when inputs changed
# or when you intentionally want to start over.
FORCE_UNZIP = False
FORCE_CONVERT = False
# Optional task-based pruning after Step 6. Dry-run unless APPLY_TASK_PRUNE=True.
RUN_TASK_PRUNE = False
APPLY_TASK_PRUNE = False
PRUNE_TASKS = []  # Exact task strings to delete, for example ['pick up the red pepper'].
PRUNE_TASK_FILE = None  # Optional Drive path to a text file with one exact task per line.
PRUNE_TASK_CONTAINS = ['largest', 'smallest']  # Substring fallback; set [] to disable.
PRUNE_TRASH_DIR = DRIVE_ROOT / 'pruned_lerobot_backup'
FORCE_NORM_STATS = False
FORCE_RESTART_TRAINING = False
RUN_SMOKE_TRAIN = False
# Use 'offline' for stable unattended Colab runs. Change to 'online' after running wandb.login().
WANDB_MODE_SETTING = 'offline'

for p in [DRIVE_ROOT, RAW_ROOT, CODE_ROOT, HF_LEROBOT_HOME, PI05_BASE_PARAMS.parent, ASSETS_BACKUP, CHECKPOINT_BACKUP, FINAL_MODELS, STATE_DIR]:
    p.mkdir(parents=True, exist_ok=True)
Path('/content/uv_cache').mkdir(parents=True, exist_ok=True)
Path('/content/pip_cache').mkdir(parents=True, exist_ok=True)

os.environ['HF_LEROBOT_HOME'] = str(HF_LEROBOT_HOME)
os.environ['UV_CACHE_DIR'] = '/content/uv_cache'
os.environ['PIP_CACHE_DIR'] = '/content/pip_cache'
os.environ['UV_LINK_MODE'] = 'copy'
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.9'
os.environ['WANDB_MODE'] = WANDB_MODE_SETTING
for key in ('WANDB_SERVICE', 'WANDB_SERVICE_TOKEN'):
    os.environ.pop(key, None)

def run(cmd, *, cwd=None, env=None):
    print('+', ' '.join(map(str, cmd)))
    return subprocess.run(list(map(str, cmd)), cwd=cwd, env=env, check=True)

def manifest_for(paths):
    h = hashlib.sha256()
    for root in paths:
        root = Path(root)
        if not root.exists():
            continue
        if root.is_file():
            files = [root]
            base = root.parent
        else:
            files = sorted(p for p in root.rglob('*') if p.is_file())
            base = root
        for p in files:
            rel = p.relative_to(base).as_posix()
            st = p.stat()
            h.update(rel.encode())
            h.update(str(st.st_size).encode())
            h.update(str(int(st.st_mtime)).encode())
    return h.hexdigest()



def raw_data_manifest(raw_root: Path):
    # Keep Colab reruns fast: use cheap Drive metadata and the explicit
    # RAW_DATA_VERSION knob instead of walking every image or episode file.
    h = hashlib.sha256()
    h.update(str(RAW_DATA_VERSION).encode())
    h.update(f'raw_root:{raw_root.as_posix()}'.encode())
    if RAW_ZIP.exists():
        st = RAW_ZIP.stat()
        h.update(f'raw_zip:{st.st_size}:{int(st.st_mtime)}'.encode())
    if raw_root.exists():
        for p in sorted(raw_root.glob('*.zip')):
            st = p.stat()
            h.update(f'task_zip:{p.name}:{st.st_size}:{int(st.st_mtime)}'.encode())
        for task_dir in sorted(p for p in raw_root.iterdir() if p.is_dir()):
            st = task_dir.stat()
            h.update(f'task_dir:{task_dir.name}:{int(st.st_mtime)}'.encode())
    return h.hexdigest()

def read_json(path, default=None):
    path = Path(path)
    if not path.exists():
        return default
    return json.loads(path.read_text())

def write_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, sort_keys=True) + '\n')

print('Drive root:', DRIVE_ROOT)
print('Raw root:', RAW_ROOT, 'exists:', RAW_ROOT.exists())
print('Raw zip:', RAW_ZIP, 'exists:', RAW_ZIP.exists())
print('Code root files:', sorted(p.name for p in CODE_ROOT.glob('*')))
print('Checkpoint backup:', CHECKPOINT_BACKUP)


## 2. Check GPU

In [ ]:
!nvidia-smi

## 3. Prepare Raw Data and Project Code

Raw data is extracted from either one `raw.zip` containing `raw/`, or per-task zip files placed directly under Drive `raw/` such as `raw/pick_up_the_red_pepper.zip`. Extracted task folders are verified and copied into `/content/embodied-ai-xarm` each runtime because `/content` is temporary.

In [ ]:
required_code_files = [
    'convert_xarm_raw_to_lerobot.py',
    'openpi_xarm_config.py',
    'xarm_data_config.py',
]
optional_code_files = [
    'delete_lerobot_task_parquets.py',
]
for filename in required_code_files:
    src = CODE_ROOT / filename
    assert src.exists(), f'Missing {src}. Copy it to Drive code/ first.'

project_ft = PROJECT_DIR / 'fine_tune'
project_ft.mkdir(parents=True, exist_ok=True)
for filename in required_code_files:
    shutil.copy2(CODE_ROOT / filename, project_ft / filename)
for filename in optional_code_files:
    src = CODE_ROOT / filename
    if src.exists():
        shutil.copy2(src, project_ft / filename)
    else:
        print(f'Optional code file not found on Drive, skipping: {src}')

print('Copied code to:', project_ft)
print('Code files:', sorted(p.name for p in project_ft.glob('*.py')))



def count_episodes(raw_root: Path):
    tasks = {}
    if raw_root.exists():
        for task_dir in sorted(p for p in raw_root.iterdir() if p.is_dir()):
            tasks[task_dir.name] = len(list(task_dir.glob('episode_*')))
    meta_count = len(list(raw_root.glob('*/*/meta.json'))) if raw_root.exists() else 0
    log_count = len(list(raw_root.glob('*/*/robot_log.csv'))) if raw_root.exists() else 0
    return tasks, meta_count, log_count

def print_counts(raw_root: Path, title: str):
    print(title)
    tasks, meta_count, log_count = count_episodes(raw_root)
    for task, count in tasks.items():
        print(f'{task}: {count} episodes')
    print('meta count:', meta_count)
    print('robot_log count:', log_count)
    return tasks, meta_count, log_count

def task_zip_paths(raw_root: Path):
    return sorted(p for p in raw_root.glob('*.zip') if p.is_file())

def copy_task_tree(src: Path, dst: Path):
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)

def install_task_zip(zip_path: Path, local_raw: Path):
    extract_dir = LOCAL_EXTRACT / f'_extract_{zip_path.stem}'
    shutil.rmtree(extract_dir, ignore_errors=True)
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(extract_dir)

    raw_inside = extract_dir / 'raw'
    search_root = raw_inside if raw_inside.exists() else extract_dir
    top_dirs = sorted(p for p in search_root.iterdir() if p.is_dir())
    installed = []

    # Case A: the zip root is one task folder or several task folders.
    for candidate in top_dirs:
        if list(candidate.glob('episode_*')):
            dst = local_raw / candidate.name
            copy_task_tree(candidate, dst)
            installed.append(dst)

    # Case B: the zip root itself contains episode_*/ folders.
    if not installed and list(search_root.glob('episode_*')):
        dst = local_raw / zip_path.stem
        copy_task_tree(search_root, dst)
        installed.append(dst)

    if not installed:
        raise RuntimeError(
            f'Could not find task episodes in {zip_path}. Expected either '
            'task_name/episode_*/..., raw/task_name/episode_*/..., or episode_*/... inside the zip.'
        )

    shutil.rmtree(extract_dir, ignore_errors=True)
    print(f'Extracted {zip_path.name}:', ', '.join(p.name for p in installed))
    return installed

EXPECTED_EPISODES = 290
LOCAL_EXTRACT = Path('/content/raw_extract_check')
LOCAL_RAW = LOCAL_EXTRACT / 'raw'

_, drive_meta_count, drive_log_count = count_episodes(RAW_ROOT)
raw_task_zips = task_zip_paths(RAW_ROOT)
raw_source_available = RAW_ZIP.exists() or bool(raw_task_zips) or drive_meta_count > 0 or drive_log_count > 0
need_extract = raw_source_available and (
    FORCE_UNZIP
    or drive_meta_count != EXPECTED_EPISODES
    or drive_log_count != EXPECTED_EPISODES
)

if not raw_source_available:
    print('No raw data source found on Drive; skipping raw verification/extraction.')
    print('This is OK when Step 6 uses an already-uploaded LeRobot dataset:', HF_LEROBOT_HOME / REPO_ID)
elif need_extract:
    print('Extracting raw data to local disk first...')
    shutil.rmtree(LOCAL_EXTRACT, ignore_errors=True)
    LOCAL_RAW.mkdir(parents=True, exist_ok=True)

    if RAW_ZIP.exists():
        print('Using monolithic raw zip:', RAW_ZIP)
        with zipfile.ZipFile(RAW_ZIP) as zf:
            zf.extractall(LOCAL_EXTRACT)
        assert LOCAL_RAW.exists(), f'Expected raw folder inside zip: {LOCAL_RAW}'
    else:
        print('Using per-task zip files under:', RAW_ROOT)
        for zip_path in raw_task_zips:
            install_task_zip(zip_path, LOCAL_RAW)

    _, meta_count, log_count = print_counts(LOCAL_RAW, 'Local extracted raw:')
    assert meta_count == EXPECTED_EPISODES, f'Expected {EXPECTED_EPISODES} meta.json files, got {meta_count}'
    assert log_count == EXPECTED_EPISODES, f'Expected {EXPECTED_EPISODES} robot_log.csv files, got {log_count}'

    if RAW_ZIP.exists():
        print('Replacing Drive raw with verified local extraction...')
        shutil.rmtree(RAW_ROOT, ignore_errors=True)
        RAW_ROOT.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run([
            'rsync', '-ah', '--delete', '--info=progress2',
            f'{LOCAL_RAW}/',
            f'{RAW_ROOT}/',
        ], check=True)
    else:
        print('Syncing extracted task folders to Drive raw while preserving task zip files...')
        RAW_ROOT.mkdir(parents=True, exist_ok=True)
        for task_dir in sorted(p for p in LOCAL_RAW.iterdir() if p.is_dir()):
            dst = RAW_ROOT / task_dir.name
            dst.mkdir(parents=True, exist_ok=True)
            subprocess.run([
                'rsync', '-ah', '--delete', '--info=progress2',
                f'{task_dir}/',
                f'{dst}/',
            ], check=True)

    _, meta_count, log_count = print_counts(RAW_ROOT, 'Drive raw after sync:')
    assert meta_count == EXPECTED_EPISODES, f'Drive raw incomplete after sync: {meta_count}'
    assert log_count == EXPECTED_EPISODES, f'Drive raw incomplete after sync: {log_count}'

    shutil.rmtree(LOCAL_EXTRACT, ignore_errors=True)
    print('Removed local temp raw extract:', LOCAL_EXTRACT)
else:
    print('Using existing verified raw data on Drive:', RAW_ROOT)
    if raw_task_zips:
        print('Task zip files present:', [p.name for p in raw_task_zips])
    print_counts(RAW_ROOT, 'Drive raw:')


## 4. Clone and Install OpenPI

This runs after every new runtime because `/content/openpi` and its virtualenv are not persistent.

In [ ]:
%%bash
set -e
cd /content
if [ ! -d openpi ]; then
  git clone --recurse-submodules https://github.com/Physical-Intelligence/openpi.git
fi
cd /content/openpi
pip install -q uv
export UV_CACHE_DIR=/content/uv_cache
export UV_LINK_MODE=copy
GIT_LFS_SKIP_SMUDGE=1 uv sync
GIT_LFS_SKIP_SMUDGE=1 uv pip install -e .

## 5. Patch OpenPI Config and Keep Checkpoints Local During Training

This cell is idempotent. It inserts the xArm config only if missing, points the base checkpoint loader at Drive, and makes sure `/content/openpi/checkpoints` is a normal local directory. Step 9 syncs completed numeric checkpoints between this local directory and Drive.


In [ ]:
import re
import runpy

openpi_config = OPENPI_DIR / 'src/openpi/training/config.py'
snippet_file = PROJECT_DIR / 'fine_tune/openpi_xarm_config.py'
snippet = runpy.run_path(str(snippet_file))['XARM_CONFIG_SNIPPET']
snippet = snippet.replace('name="pi05_xarm_full_finetune"', f'name="{CONFIG_NAME}"', 1)
snippet = snippet.replace('name="pi05_xarm_colab_smoke"', f'name="{SMOKE_CONFIG_NAME}"', 1)
snippet = snippet.replace('repo_id="local/xarm_pi05_data"', f'repo_id="{REPO_ID}"')
snippet = snippet.replace(
    'weight_loader=weight_loaders.CheckpointWeightLoader("gs://openpi-assets/checkpoints/pi05_base/params")',
    f'weight_loader=weight_loaders.CheckpointWeightLoader("{PI05_BASE_PARAMS.as_posix()}")',
)
marker = '# Add these TrainConfig entries to _CONFIGS.'
class_part, train_tail = snippet.split(marker, 1)
train_part = marker + train_tail

def remove_train_config_named(text: str, name: str) -> str:
    match = re.search(r'TrainConfig\(\s*\n\s*name="' + re.escape(name) + r'"', text)
    if not match:
        return text
    start = match.start()
    depth = 0
    i = start
    while i < len(text):
        if text[i] == '(':
            depth += 1
        elif text[i] == ')':
            depth -= 1
            if depth == 0:
                end = i + 1
                while end < len(text) and text[end] in ', \t\r\n':
                    end += 1
                return text[:start].rstrip() + '\n\n' + text[end:].lstrip()
        i += 1
    raise RuntimeError(f'Could not find end of TrainConfig named {name}')

if 'pi05_xarm' not in {CONFIG_NAME, SMOKE_CONFIG_NAME}:
    train_part = remove_train_config_named(train_part, 'pi05_xarm')

# Keep your current experiment name stable even if the Drive snippet only has
# the older pi05_xarm config entry. Clone only the first TrainConfig block.
if CONFIG_NAME not in train_part and 'name="pi05_xarm"' in train_part:
    first_entry_body = train_tail.split('TrainConfig(', 1)[1].split('\n\nTrainConfig(', 1)[0]
    full_entry = 'TrainConfig(' + first_entry_body
    full_entry = full_entry.replace('name="pi05_xarm"', f'name="{CONFIG_NAME}"', 1)
    full_entry = full_entry.replace('num_train_steps=30_000', 'num_train_steps=20_000', 1)
    train_part = marker + '\n' + full_entry + '\n\n' + train_tail.lstrip()

text = openpi_config.read_text()
if 'import pathlib' not in text:
    text = text.replace('import dataclasses\n', 'import dataclasses\nimport pathlib\n', 1)
if 'override' not in text:
    text = text.replace('import pathlib\n', 'import pathlib\nfrom typing_extensions import override\n', 1)
if 'import openpi.policies.libero_policy as libero_policy' not in text:
    text = text.replace(
        'import openpi.policies.aloha_policy as aloha_policy\n',
        'import openpi.policies.aloha_policy as aloha_policy\nimport openpi.policies.libero_policy as libero_policy\n',
    )
if 'class LeRobotXArmDataConfig' not in text:
    text = text.replace('_CONFIGS = [', class_part.rstrip() + '\n\n_CONFIGS = [', 1)
for stale_config_name in sorted({CONFIG_NAME, SMOKE_CONFIG_NAME, 'pi05_xarm_full_finetune', 'pi05_xarm', 'pi05_xarm_colab_smoke'}):
    text = remove_train_config_named(text, stale_config_name)
text = text.replace('_CONFIGS = [', '_CONFIGS = [\n' + train_part.rstrip() + '\n', 1)

text = text.replace(
    'weight_loader=weight_loaders.CheckpointWeightLoader("gs://openpi-assets/checkpoints/pi05_base/params")',
    f'weight_loader=weight_loaders.CheckpointWeightLoader("{PI05_BASE_PARAMS.as_posix()}")',
)
text = text.replace('repo_id="local/xarm_pi05_data"', f'repo_id="{REPO_ID}"')
openpi_config.write_text(text)

# Make W&B use notebook environment variables instead of OpenPI's default
# project name, so metrics are easy to find in the intended dashboard.
train_script = OPENPI_DIR / 'scripts/train.py'
train_text = train_script.read_text()
if 'import os\n' not in train_text:
    if train_text.startswith('from __future__ import annotations\n'):
        train_text = train_text.replace('from __future__ import annotations\n', 'from __future__ import annotations\nimport os\n', 1)
    else:
        train_text = 'import os\n' + train_text
train_text = train_text.replace('project="openpi"', 'project=os.environ.get("WANDB_PROJECT", "openpi")')
train_text = train_text.replace('project="openpi",', 'project=os.environ.get("WANDB_PROJECT", "openpi"),')
train_text = train_text.replace('name=name,', 'name=os.environ.get("WANDB_NAME", name),')
train_script.write_text(train_text)

# Disable OpenPI's tqdm-style progress logger in Colab. It emits repeated
# "Progress on: ..." lines through openpi/shared/tqdm_logging.py, which makes
# notebook logs hard to read. Scalar metrics such as loss still print normally,
# and W&B receives the training metrics.
tqdm_logging_file = OPENPI_DIR / 'src/openpi/shared/tqdm_logging.py'
if tqdm_logging_file.exists():
    tqdm_text = tqdm_logging_file.read_text()
    marker = '# CODEX_COLAB_DISABLE_TQDM_PROGRESS\n'
    if marker not in tqdm_text:
        tqdm_text = tqdm_text.rstrip() + '\n\n' + marker + 'def log_tqdm(*args, **kwargs):\n    return None\n'
        tqdm_logging_file.write_text(tqdm_text)
        print('Disabled OpenPI tqdm progress logging:', tqdm_logging_file)
    else:
        print('OpenPI tqdm progress logging already disabled:', tqdm_logging_file)
else:
    print('OpenPI tqdm logging file not found; skipping:', tqdm_logging_file)

# Orbax/TensorStore checkpoints are fragile on Google Drive mounts because they
# rely on many lock files and atomic renames. Keep the live checkpoint directory
# local, then sync completed numeric checkpoints to Drive in Step 9.
local_ckpts = OPENPI_DIR / 'checkpoints'
if local_ckpts.is_symlink():
    local_ckpts.unlink()
elif local_ckpts.exists() and not local_ckpts.is_dir():
    local_ckpts.unlink()
local_ckpts.mkdir(parents=True, exist_ok=True)
CHECKPOINT_BACKUP.mkdir(parents=True, exist_ok=True)

print('OpenPI config patched:', openpi_config)
print('Local checkpoint directory:', local_ckpts)
print('Drive checkpoint backup:', CHECKPOINT_BACKUP)
grep_pattern = f'LeRobotXArmDataConfig|{CONFIG_NAME}|{SMOKE_CONFIG_NAME}|{REPO_ID}|openpi_cache/pi05_base/params'
!grep -nE "{grep_pattern}" /content/openpi/src/openpi/training/config.py


In [ ]:
import re

def train_config_block_named(text: str, name: str) -> str | None:
    match = re.search(r'TrainConfig\(\s*\n\s*name="' + re.escape(name) + r'"', text)
    if not match:
        return None
    start = match.start()
    depth = 0
    i = start
    while i < len(text):
        if text[i] == '(':
            depth += 1
        elif text[i] == ')':
            depth -= 1
            if depth == 0:
                end = i + 1
                if end < len(text) and text[end] == ',':
                    end += 1
                return text[start:end]
        i += 1
    raise RuntimeError(f'Could not find end of TrainConfig named {name}')

config_text = openpi_config.read_text()
for name in dict.fromkeys([CONFIG_NAME, SMOKE_CONFIG_NAME]):
    block = train_config_block_named(config_text, name)
    print(f'===== TrainConfig: {name} =====')
    if block is None:
        print('NOT FOUND')
    else:
        print(block)


## 6. Use Existing LeRobot Dataset, Convert Only If Missing

Fast path: if Drive already contains the real LeRobotDataset for `REPO_ID` with `meta/info.json` and parquet chunks, this cell trusts it and skips conversion. This is the preferred path when you converted locally and uploaded `/home/mingqian/lerobot_cache/local/xarm_pi05_data` to Drive.

Set `FORCE_CONVERT = True` in cell 1 only when you intentionally want Colab to rebuild from raw data.


In [ ]:
import time

def step6_log(message):
    print(f'[Step 6 {time.strftime("%H:%M:%S")}] {message}', flush=True)

converter_file = PROJECT_DIR / 'fine_tune/convert_xarm_raw_to_lerobot.py'
drive_dataset = Path(LEROBOT_DATASET_ROOT)
dataset_info = drive_dataset / 'meta/info.json'
convert_manifest_path = STATE_DIR / 'lerobot_conversion_manifest.json'

assert (OPENPI_DIR / 'pyproject.toml').exists(), f'Missing OpenPI checkout: {OPENPI_DIR}. Run Step 4 first.'

def first_parquet_under(dataset_root: Path):
    return next(dataset_root.glob('data/**/*.parquet'), None)

def count_parquets_limited(dataset_root: Path, limit: int = 20):
    count = 0
    for count, _ in enumerate(dataset_root.glob('data/**/*.parquet'), start=1):
        if count >= limit:
            return f'{limit}+'
    return str(count)

def cheap_dataset_manifest(dataset_root: Path):
    """Fast cache key for later norm-stat reuse; avoids hashing every image/video file."""
    tracked = [dataset_root / 'meta/info.json']
    tracked.extend(sorted((dataset_root / 'meta').glob('*.jsonl')))
    tracked.extend(sorted((dataset_root / 'meta').glob('*.json')))
    tracked.extend(sorted(dataset_root.glob('data/**/*.parquet')))
    h = hashlib.sha256()
    for p in tracked:
        if not p.exists() or not p.is_file():
            continue
        rel = p.relative_to(dataset_root).as_posix()
        st = p.stat()
        h.update(rel.encode())
        h.update(str(st.st_size).encode())
        h.update(str(int(st.st_mtime)).encode())
    return h.hexdigest()

def write_step6_manifest(source: str):
    manifest = {
        'source': source,
        'dataset': cheap_dataset_manifest(drive_dataset),
        'repo_id': REPO_ID,
        'fps': FPS,
        'robot_type': ROBOT_TYPE,
        'conversion_mode': 'uploaded_or_colab_lerobot_dataset_v2',
    }
    if read_json(convert_manifest_path, {}) == manifest:
        step6_log(f'Pipeline manifest already current: {convert_manifest_path}')
    else:
        write_json(convert_manifest_path, manifest)
        step6_log(f'Wrote pipeline manifest: {convert_manifest_path}')

step6_log(f'Drive LeRobot dataset: {drive_dataset}')
step6_log(f'Pipeline manifest: {convert_manifest_path}')
first_existing_parquet = first_parquet_under(drive_dataset)

if dataset_info.exists() and first_existing_parquet is not None and not FORCE_CONVERT:
    step6_log('Found uploaded LeRobotDataset on Drive; skipping raw copy and conversion.')
    write_step6_manifest('drive_existing')
else:
    raw_meta_count = len(list(RAW_ROOT.glob('*/*/meta.json'))) if RAW_ROOT.exists() else 0
    raw_log_count = len(list(RAW_ROOT.glob('*/*/robot_log.csv'))) if RAW_ROOT.exists() else 0
    if raw_meta_count == 0 or raw_log_count == 0:
        raise FileNotFoundError(
            'No usable converted LeRobot dataset was found, and no raw data is available for Colab conversion.\n'
            f'Expected converted dataset at: {drive_dataset}\n'
            f'Expected metadata file: {dataset_info}\n'
            f'Expected parquet files under: {drive_dataset / "data"}\n'
            'Fix one of these before rerunning Step 6:\n'
            '  1. Upload/extract your converted xarm_pi05_data folder to that exact path.\n'
            '  2. Set LEROBOT_DATASET_ROOT in Cell 1 to the actual folder containing meta/info.json.\n'
            '  3. Upload raw data if you want Colab to rebuild from raw.'
        )

    assert converter_file.exists(), f'Missing converter: {converter_file}. Run Step 3 first.'
    assert RAW_ROOT.exists(), f'Missing raw data root: {RAW_ROOT}. Run Step 3 first, or upload the converted dataset to {drive_dataset}.'
    assert 'EXPECTED_EPISODES' in globals(), 'EXPECTED_EPISODES is not defined. Run Step 3 first.'
    assert 'print_counts' in globals(), 'print_counts is not defined. Run Step 3 first.'

    local_raw = Path('/content/raw_local')
    local_hf = Path('/content/lerobot_local')
    local_light = Path('/content/lerobot_light_xarm')
    local_dataset = local_hf / REPO_ID

    if FORCE_CONVERT:
        step6_log('FORCE_CONVERT=True; rebuilding LeRobot dataset in Colab.')
    elif not dataset_info.exists():
        step6_log('Dataset metadata is missing on Drive; rebuilding LeRobot dataset.')
    else:
        step6_log('No parquet chunks found on Drive; rebuilding LeRobot dataset.')

    step6_log('Copying Drive raw to local disk for conversion...')
    shutil.rmtree(local_raw, ignore_errors=True)
    local_raw.mkdir(parents=True, exist_ok=True)
    subprocess.run(['rsync', '-ah', '--delete', '--info=progress2', f'{RAW_ROOT}/', f'{local_raw}/'], check=True)

    _, meta_count, log_count = print_counts(local_raw, 'Local raw for conversion:')
    assert meta_count == EXPECTED_EPISODES, f'Local raw incomplete: {meta_count}'
    assert log_count == EXPECTED_EPISODES, f'Local raw incomplete: {log_count}'

    step6_log('Converting locally, then syncing the finished LeRobotDataset to Drive...')
    shutil.rmtree(local_hf, ignore_errors=True)
    shutil.rmtree(local_light, ignore_errors=True)
    local_hf.mkdir(parents=True, exist_ok=True)

    env = os.environ.copy()
    env['HF_LEROBOT_HOME'] = str(local_hf)
    run([
        'uv', 'run', 'python', str(converter_file),
        '--raw-root', str(local_raw),
        '--output-dir', str(local_light),
        '--repo-id', REPO_ID,
        '--robot-type', ROBOT_TYPE,
        '--fps', str(FPS),
        '--overwrite',
        '--skip-light-image-copy',
        '--skip-hf-dataset',
    ], cwd=OPENPI_DIR, env=env)

    assert (local_dataset / 'meta/info.json').exists(), f'Local conversion missing metadata: {local_dataset}'
    local_parquets = sorted(local_dataset.glob('data/**/*.parquet'))
    assert local_parquets, f'Local conversion did not create parquet chunks under {local_dataset / "data"}'
    drive_dataset.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['rsync', '-ah', '--delete', '--info=progress2', f'{local_dataset}/', f'{drive_dataset}/'], check=True)

    shutil.rmtree(local_raw, ignore_errors=True)
    shutil.rmtree(local_light, ignore_errors=True)
    shutil.rmtree(local_hf, ignore_errors=True)

    assert dataset_info.exists(), f'Conversion finished but metadata is missing: {dataset_info}'
    assert first_parquet_under(drive_dataset) is not None, f'Drive conversion missing parquet chunks under {drive_dataset / "data"}'
    write_step6_manifest('colab_conversion')

print('Dataset info:', dataset_info)
print('Parquet chunks:', count_parquets_limited(drive_dataset))
print(dataset_info.read_text()[:1200])


## 6B. Optional Task-Based LeRobot Pruning

Run this only when you want to remove parquet files for selected tasks before norm stats. The default is a dry run. Use `PRUNE_TASKS` for exact task names, `PRUNE_TASK_FILE` for a line-based task list, or `PRUNE_TASK_CONTAINS` for substring matching. Set `RUN_TASK_PRUNE = True` and `APPLY_TASK_PRUNE = True` in cell 1 to actually modify the Drive LeRobot dataset.


In [ ]:
prune_script = PROJECT_DIR / 'fine_tune/delete_lerobot_task_parquets.py'

if RUN_TASK_PRUNE:
    assert prune_script.exists(), f'Missing {prune_script}. Copy delete_lerobot_task_parquets.py to Drive code/ and re-run Step 3.'
    assert dataset_info.exists(), f'Missing LeRobotDataset metadata: {dataset_info}. Run Step 6 first.'
    assert first_parquet_under(drive_dataset) is not None, f'Missing parquet chunks under {drive_dataset / "data"}. Run Step 6 first.'

    cmd = [
        'uv', 'run', 'python', str(prune_script),
        '--dataset-root', str(drive_dataset),
    ]
    for task in PRUNE_TASKS:
        cmd.extend(['--task', str(task)])
    if PRUNE_TASK_FILE:
        cmd.extend(['--task-file', str(PRUNE_TASK_FILE)])
    for term in PRUNE_TASK_CONTAINS:
        cmd.extend(['--task-contains', str(term)])
    if APPLY_TASK_PRUNE:
        cmd.append('--apply')
        cmd.extend(['--trash-dir', str(PRUNE_TRASH_DIR)])

    print('Task pruning mode:', 'APPLY' if APPLY_TASK_PRUNE else 'DRY RUN')
    print('Exact tasks:', PRUNE_TASKS)
    print('Task file:', PRUNE_TASK_FILE)
    print('Task contains:', PRUNE_TASK_CONTAINS)
    run(cmd, cwd=OPENPI_DIR)

    if APPLY_TASK_PRUNE:
        assert dataset_info.exists(), f'Pruning finished but metadata is missing: {dataset_info}'
        assert first_parquet_under(drive_dataset) is not None, f'Pruning removed all parquet chunks under {drive_dataset / "data"}'
        write_step6_manifest('drive_existing_pruned')
        print('Pruned dataset info:', dataset_info)
        print(dataset_info.read_text()[:1200])
else:
    print('RUN_TASK_PRUNE=False; skipping task-based LeRobot pruning.')


## 7. Cache pi05 Base Checkpoint on Drive

In [ ]:
if not (PI05_BASE_PARAMS / 'commit_success.txt').exists():
    print('Caching pi05_base params to Drive. This can take several minutes.')
    run(['pip', 'install', '--no-cache-dir', '-U', 'crcmod'])
    if PI05_BASE_PARAMS.exists():
        shutil.rmtree(PI05_BASE_PARAMS)
    PI05_BASE_PARAMS.mkdir(parents=True, exist_ok=True)
    run(['bash', '-lc', f'gsutil -m cp -r gs://openpi-assets/checkpoints/pi05_base/params/* "{PI05_BASE_PARAMS}/"'])
else:
    print('pi05_base params already cached:', PI05_BASE_PARAMS)
!ls -lah "{PI05_BASE_PARAMS}" | head

## 8. Compute or Restore Norm Stats

Norm stats are restored from Drive when the converted dataset and config are unchanged. Recompute only after changing data, converter, repo id, or OpenPI config transforms.

In [ ]:
assert dataset_info.exists(), f'Missing LeRobotDataset metadata: {dataset_info}. Run Step 6 first.'

config_file = PROJECT_DIR / 'fine_tune/openpi_xarm_config.py'
norm_manifest_path = STATE_DIR / f'norm_stats_{CONFIG_NAME}.json'
local_assets = OPENPI_DIR / 'assets' / CONFIG_NAME
backup_assets = ASSETS_BACKUP / CONFIG_NAME
current_norm_manifest = {
    'dataset_and_config': manifest_for([dataset_info, config_file, convert_manifest_path]),
    'config_name': CONFIG_NAME,
    'repo_id': REPO_ID,
}
previous_norm_manifest = read_json(norm_manifest_path, {})

if backup_assets.exists() and previous_norm_manifest == current_norm_manifest and not FORCE_NORM_STATS:
    if local_assets.exists():
        shutil.rmtree(local_assets)
    local_assets.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(backup_assets, local_assets)
    print('Restored norm stats from Drive:', backup_assets)
else:
    if local_assets.exists():
        shutil.rmtree(local_assets)
    if backup_assets.exists():
        shutil.rmtree(backup_assets)
    run(['uv', 'run', 'scripts/compute_norm_stats.py', '--config-name', CONFIG_NAME], cwd=OPENPI_DIR)
    assert local_assets.exists(), f'Norm stats were not created: {local_assets}'
    shutil.copytree(local_assets, backup_assets)
    write_json(norm_manifest_path, current_norm_manifest)
    print('Computed and backed up norm stats:', backup_assets)

!find "{local_assets}" -maxdepth 3 -type f | sort

## 9. Release Runtime and Local Checkpoint Disk Space

Run this cell after stopping/interruption or before starting a new experiment in the same Colab runtime. It clears stale runtime state and frees Colab disk space by deleting local checkpoint copies only when the same complete checkpoint already exists in Drive.

In [ ]:
import gc
import os
import signal
import shutil
import subprocess
import time
from pathlib import Path

print('Releasing runtime state and local checkpoint disk space...')

for key in ('WANDB_SERVICE', 'WANDB_SERVICE_TOKEN'):
    os.environ.pop(key, None)

try:
    import wandb
    wandb.finish()
    print('Finished active W&B run, if any.')
except Exception as exc:
    print('W&B cleanup skipped:', repr(exc))

def pids_matching(pattern: str):
    result = subprocess.run(['pgrep', '-f', pattern], capture_output=True, text=True)
    if result.returncode not in (0, 1):
        return []
    current_pid = os.getpid()
    parent_pid = os.getppid()
    pids = []
    for line in result.stdout.splitlines():
        try:
            pid = int(line.strip())
        except ValueError:
            continue
        if pid not in (current_pid, parent_pid):
            pids.append(pid)
    return sorted(set(pids))

patterns = ['scripts/train.py', 'openpi.training', 'wandb-service']
terminated = []
for pattern in patterns:
    for pid in pids_matching(pattern):
        try:
            os.kill(pid, signal.SIGTERM)
            terminated.append((pid, pattern))
        except ProcessLookupError:
            pass
        except PermissionError as exc:
            print(f'Could not terminate pid {pid} for {pattern}: {exc}')

if terminated:
    print('Sent SIGTERM to leftover processes:', terminated)
    time.sleep(5)
else:
    print('No leftover train/W&B service processes found.')

for pattern in patterns:
    for pid in pids_matching(pattern):
        try:
            os.kill(pid, signal.SIGKILL)
            print(f'Sent SIGKILL to stubborn pid {pid} for {pattern}')
        except ProcessLookupError:
            pass
        except PermissionError as exc:
            print(f'Could not kill pid {pid} for {pattern}: {exc}')

def checkpoint_is_complete(path: Path):
    return path.is_dir() and (path / '_CHECKPOINT_METADATA').exists() and (path / 'assets').exists() and (path / 'params').exists() and (path / 'train_state').exists()

def complete_checkpoint_steps(run_dir: Path):
    if not run_dir.exists():
        return []
    return sorted(int(p.name) for p in run_dir.iterdir() if p.name.isdigit() and checkpoint_is_complete(p))

local_run_dir = OPENPI_DIR / 'checkpoints' / CONFIG_NAME / EXP_NAME
drive_run_dir = CHECKPOINT_BACKUP / CONFIG_NAME / EXP_NAME
local_steps = complete_checkpoint_steps(local_run_dir)
drive_steps = set(complete_checkpoint_steps(drive_run_dir))
print('Local complete checkpoint steps:', local_steps)
print('Drive complete checkpoint steps:', sorted(drive_steps))

freed_steps = []
for step in local_steps:
    if step in drive_steps:
        local_step_dir = local_run_dir / str(step)
        shutil.rmtree(local_step_dir)
        freed_steps.append(step)
    else:
        print(f'Keeping local checkpoint {step}; no complete Drive copy found.')
print('Deleted local checkpoint copies already backed up to Drive:', freed_steps)

if local_run_dir.exists():
    for tmp_dir in list(local_run_dir.glob('.orbax-checkpoint-tmp-*')) + list(local_run_dir.glob('tmp*')):
        if tmp_dir.is_dir():
            print('Deleting incomplete local temp checkpoint:', tmp_dir)
            shutil.rmtree(tmp_dir, ignore_errors=True)
    if not any(local_run_dir.iterdir()):
        shutil.rmtree(local_run_dir)
        print('Removed empty local run dir:', local_run_dir)

try:
    import jax
    jax.clear_caches()
    print('Cleared JAX caches.')
except Exception as exc:
    print('JAX cache cleanup skipped:', repr(exc))

try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        print('Cleared PyTorch CUDA cache.')
except Exception as exc:
    print('PyTorch CUDA cleanup skipped:', repr(exc))

gc.collect()
print('Python garbage collection complete.')
subprocess.run(['df', '-h', '/content'], check=False)
subprocess.run(['du', '-sh', str(OPENPI_DIR / 'checkpoints')], check=False)
subprocess.run(['nvidia-smi'], check=False)


## 10. Train or Resume Locally, Then Sync Checkpoints to Drive

This is the main one-click training cell. It restores completed numeric checkpoints from Drive into local `/content/openpi/checkpoints`, trains against the local directory, then syncs completed checkpoints back to Drive. This avoids Orbax/TensorStore save failures caused by Google Drive mount rename/lock-file behavior.


In [ ]:
def complete_checkpoint_steps(run_dir: Path):
    if not run_dir.exists():
        return []
    steps = []
    for p in run_dir.iterdir():
        if p.is_dir() and p.name.isdigit() and (p / '_CHECKPOINT_METADATA').exists() and (p / 'assets').exists() and (p / 'params').exists() and (p / 'train_state').exists():
            steps.append(int(p.name))
    return sorted(steps)

local_root = OPENPI_DIR / 'checkpoints'
local_run_dir = local_root / CONFIG_NAME / EXP_NAME
drive_run_dir = CHECKPOINT_BACKUP / CONFIG_NAME / EXP_NAME
local_root.mkdir(parents=True, exist_ok=True)
drive_run_dir.mkdir(parents=True, exist_ok=True)

if local_root.is_symlink():
    raise RuntimeError(f'{local_root} is still a symlink. Re-run Step 5 before training.')

drive_steps = complete_checkpoint_steps(drive_run_dir)
print('Local checkpoint run dir:', local_run_dir)
print('Drive checkpoint backup dir:', drive_run_dir)
print('Existing complete Drive checkpoint steps:', drive_steps)

if FORCE_RESTART_TRAINING:
    print('FORCE_RESTART_TRAINING=True: wiping local and Drive run dirs and starting over.')
    if local_run_dir.exists():
        shutil.rmtree(local_run_dir)
    if drive_run_dir.exists():
        shutil.rmtree(drive_run_dir)
    drive_run_dir.mkdir(parents=True, exist_ok=True)
    mode_arg = '--overwrite'
elif drive_steps:
    print('Restoring complete checkpoints from Drive to local before resume.')
    if local_run_dir.exists():
        shutil.rmtree(local_run_dir)
    local_run_dir.mkdir(parents=True, exist_ok=True)
    for step in drive_steps:
        src = drive_run_dir / str(step)
        dst = local_run_dir / str(step)
        run(['rsync', '-ah', '--delete', f'{src.as_posix()}/', f'{dst.as_posix()}/'])
    print('Resuming from latest checkpoint:', drive_steps[-1])
    mode_arg = '--resume'
else:
    print('No complete Drive checkpoint found; starting a new run.')
    if local_run_dir.exists():
        shutil.rmtree(local_run_dir)
    mode_arg = '--overwrite'

if RUN_SMOKE_TRAIN:
    run(['uv', 'run', 'scripts/train.py', SMOKE_CONFIG_NAME, '--exp-name=xarm_smoke', '--overwrite'], cwd=OPENPI_DIR)
else:
    env = os.environ.copy()
    env['WANDB_MODE'] = WANDB_MODE_SETTING
    env['WANDB_PROJECT'] = 'embodied_ai_xarm'
    env['WANDB_NAME'] = EXP_NAME
    env['WANDB_CONSOLE'] = 'wrap'
    env['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.9'
    env['PYTHONUNBUFFERED'] = '1'
    for key in ('WANDB_SERVICE', 'WANDB_SERVICE_TOKEN'):
        env.pop(key, None)
    try:
        run(['uv', 'run', 'python', '-u', 'scripts/train.py', CONFIG_NAME, f'--exp-name={EXP_NAME}', mode_arg], cwd=OPENPI_DIR, env=env)
    finally:
        local_steps = complete_checkpoint_steps(local_run_dir)
        print('Complete local checkpoint steps after training attempt:', local_steps)
        drive_run_dir.mkdir(parents=True, exist_ok=True)
        for step in local_steps:
            src = local_run_dir / str(step)
            dst = drive_run_dir / str(step)
            print('Syncing checkpoint to Drive:', step)
            run(['rsync', '-ah', '--delete', f'{src.as_posix()}/', f'{dst.as_posix()}/'])
        print('Complete Drive checkpoint steps after sync:', complete_checkpoint_steps(drive_run_dir))

## 11. Export Latest Complete Checkpoint for Inference

The export chooses the latest complete numeric checkpoint containing `_CHECKPOINT_METADATA`, `assets/`, and `params/`, and omits `train_state/` to avoid carrying optimizer state into the inference package. Temporary `.orbax-checkpoint-tmp-*` directories are ignored.


In [ ]:
%%bash
set -e

CONFIG_NAME="pi05_xarm_full_finetune"
EXP_NAME="pi05_xarm_full_finetune"
RUN_DIR="/content/drive/MyDrive/embodied_ai_xarm/openpi_checkpoints/${CONFIG_NAME}/${EXP_NAME}"
FINAL_ROOT="/content/drive/MyDrive/embodied_ai_xarm/final_models/${CONFIG_NAME}_${EXP_NAME}"
TMP="/content/${CONFIG_NAME}_export"

LATEST=$(find "$RUN_DIR" -maxdepth 1 -type d -regex '.*/[0-9]+' 2>/dev/null \
  | while read d; do
      test -f "$d/_CHECKPOINT_METADATA" && test -d "$d/assets" && test -d "$d/params" && echo "$(basename "$d")"
    done \
  | sort -n | tail -1)

if [ -z "$LATEST" ]; then
  echo "No complete numeric checkpoint found under $RUN_DIR"
  exit 1
fi

SRC="$RUN_DIR/$LATEST"
DST="$FINAL_ROOT/$LATEST"
TAR="/content/drive/MyDrive/embodied_ai_xarm/${CONFIG_NAME}_${LATEST}_inference.tar.gz"

echo "Exporting checkpoint step: $LATEST"
rm -rf "$TMP" "$DST"
mkdir -p "$TMP/$LATEST" "$FINAL_ROOT"
rsync -ah --delete --exclude train_state "$SRC/" "$TMP/$LATEST/"
test -d "$TMP/$LATEST/assets"
test -d "$TMP/$LATEST/params"
rsync -ah --delete "$TMP/$LATEST/" "$DST/"
tar -czf "$TAR" -C "$TMP" "$LATEST"

cat > "$FINAL_ROOT/README.txt" <<EOF
OpenPI xArm pi0.5 inference export.

Config name: $CONFIG_NAME
Experiment name: $EXP_NAME
Checkpoint step: $LATEST

Serve with:
uv run scripts/serve_policy.py policy:checkpoint \\
  --policy.config=$CONFIG_NAME \\
  --policy.dir=$DST
EOF

echo "Exported directory: $DST"
echo "Exported archive: $TAR"
ls -lh "$TAR"
